In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle

import sys
sys.path.append('../../../src/')

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *

import config
i = 1j

path = "../../../scripts/Results/Blow_up/As/"
path_save = "Img/Heaven/As/"
path_save_local = "Img/Local/"
path_double = "../../../scripts/Results/Blow_up/As/Double_time/"

# Blow up differents A's

In [ ]:
As = [0.625, 0.63125, 0.638, 0.64375, 0.65, 0.65625, 0.6609375, 0.665625, 0.6703125, 0.675]

for j in range(1, len(As)):
    print(As[j] - As[j-1])

In [ ]:
As = [0.625, 0.63125, 0.638, 0.64375, 0.65, 0.65625, 0.6609375, 0.665625, 0.6703125, 0.675]
file_names = ["first", "second", "third", "fourth", "fifth"]

norms_inf = []
H1_norms = []
H2_norms = np.full(len(As), fill_value=None)
energies = []
masses = []
times = []

sols_A_1 = []
sols_A_2 = []

for j in range(len(As)):
    file_name = file_names[int(j/2)]
    folder = path + "blow_up_N_160000_Nt_64221_sigma_0.1_{}/".format(file_name)
    folder_name = folder + "blow_up_A_{}/".format(round(As[j], 3))

    H1_file = gzip.GzipFile(folder_name + "H1_norm.npy.gz", "r")
    H1_norms += [np.load(H1_file)]

    #if As[j]!= 0.64375:
    H2_file = gzip.GzipFile(folder_name + "H2_norm.npy.gz", "r")
    H2_norms[j] = np.load(H2_file)

    norm_file = gzip.GzipFile(folder_name + "norm_inf.npy.gz", "r")
    norms_inf += [np.load(norm_file)]

    energy_file = gzip.GzipFile(folder_name + "energy.npy.gz", "r")
    energies += [np.load(energy_file)]

    mass_file = gzip.GzipFile(folder_name + "mass.npy.gz", "r")
    masses += [np.load(mass_file)]

    time_file = gzip.GzipFile(folder_name + "time.npy.gz", "r")
    times += [np.load(time_file)]

In [ ]:
sol_lim = np.sqrt(2)/2

for j in range(len(As)):
    norm_inf_plot = norms_inf[j]
    time_plot = times[j]
    cond = norm_inf_plot!=0
    norm_inf_plot = norm_inf_plot[cond]

    cond_1_markers = np.where(norm_inf_plot<sol_lim)[0]
    mark_indices_1 = cond_1_markers[::3000]
    cond_2_markers = np.where(norm_inf_plot>sol_lim)[0]
    mark_indices_2 = cond_2_markers

    mark_indices = np.append(mark_indices_1, mark_indices_2)
    
    plt.plot(time_plot[cond], norm_inf_plot,label="A={}".format(round(As[j], 3)), markevery=mark_indices, markerfacecolor='none', ms=3.5) #

plt.xlabel(r"$t$")
plt.ylabel(r"$||\Psi||_\infty$", rotation='horizontal', ha='right')
plt.hlines(sol_lim, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.ylim(0.6, 1)
plt.xlim(0, 3.5e-3)
plt.legend(fontsize=6)
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
#plt.savefig(path_save+"Blow_up_As_inf_norm.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_inf_norm.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
sol_lim = np.sqrt(2)/2

fig = plt.figure()
#fig.patch.set_alpha(0)
ax = fig.add_subplot(111)

# Guardamos los handles y los valores de A para ordenar la leyenda al final
handles = []
A_values = []

for j in range(2, len(As)):
    norm_inf = norms_inf[j]

    time_plot = times[j]
    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)  
    norm_inf = norm_inf[cond]

    time_plot = times[j][cond]

    markers_on = np.arange(0, len(time_plot), 1500)
    markers_on = np.append(markers_on, np.where(cond)[0][-1])

    print("A:", As[j])
    print("sqrt(2)/2 - norm_inf:", -norm_inf[-1] + np.sqrt(2)/2)

    line, = ax.plot(time_plot, norm_inf, markevery=markers_on, markerfacecolor='none', ms=3.5)
    handles.append(line)
    A_values.append(As[j])

for j in range(2):
    norm_inf = norms_inf[j]
    time_plot = times[j]

    line, = ax.plot(time_plot, norm_inf, markevery=1500, markerfacecolor='none', ms=3.5)
    handles.append(line)
    A_values.append(As[j])

ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$||\Psi||_\infty$", rotation='horizontal', ha='right')
sol_line = ax.hlines(sol_lim, 0, 3.5e-3, ls="--", color="gray")
ax.set_ylim(0.6, sol_lim*1.05)
ax.set_xlim(0, 3.5e-3)

# Ordenamos la leyenda por A decreciente (sqrt(2)/2 al final)
order = np.argsort(A_values)[::-1]
handles_sorted = [handles[i] for i in order] + [sol_line]
labels_sorted = ["A={}".format(round(A_values[i], 3)) for i in order] + [r"$\sqrt{2}/2$"]
ax.legend(handles_sorted, labels_sorted, fontsize=6, bbox_to_anchor=(1.05, 1))

ax.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.savefig(path_save+"Blow_up_As_inf_norm_cut.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_inf_norm.pdf", dpi=300, bbox_inches = "tight")
plt.show()

Añadir al latex

In [ ]:
sol_lim = np.sqrt(2)/2+0.000085

for j in range(len(As)):
    #if As[j]!= 0.64375:
    norm_inf = norms_inf[j]
    cond_inf = norm_inf!=0
    norm_inf = norm_inf[cond_inf]

    cond_1_markers = np.where(norm_inf<sol_lim)[0]
    mark_indices_1 = cond_1_markers[::3000]
    cond_2_markers = np.where(norm_inf>sol_lim)[0]
    mark_indices_2 = cond_2_markers
    mark_indices = np.append(mark_indices_1, mark_indices_2)

    H2_norm_plot = H2_norms[j]
    time_plot = times[j]
    cond = H2_norm_plot!=0
    H2_norm_plot = H2_norm_plot[cond]
    plt.plot(time_plot[cond], H2_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markevery=mark_indices, markerfacecolor='none') #

plt.xlabel(r"$t$")
plt.ylabel(r"$|\Psi|_{H^2_h}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.ylim(10, 300)
plt.xlim(0, 3.5e-3)

#plt.yscale('log')
plt.legend(fontsize=6)
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

#plt.savefig(path_save+"Blow_up_As_H2_norm.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H2_norm.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
sol_lim = np.sqrt(2)/2+0.000085

fig = plt.figure()
#fig.patch.set_alpha(0)
ax = fig.add_subplot(111)

for j in range(2, len(As)):
    norm_inf = norms_inf[j]

    time_plot = times[j]
    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)  
    norm_inf = norm_inf[cond]

    H2_norm_plot = H2_norms[j][cond]

    #cond_1_markers = np.where(norm_inf_plot<sol_lim)[0]
    #mark_indices_1 = cond_1_markers[::3000]
    #cond_2_markers = np.where(norm_inf_plot>sol_lim)[0]
    #mark_indices_2 = cond_2_markers

    #mark_indices = np.append(mark_indices_1, mark_indices_2)

    #ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    #T = times[j][cond]#times[j][ind_exp]

    #mask = times[j] < T 
    time_plot = times[j][cond]#[mask]

    markers_on = np.arange(0, len(time_plot), 1500)
    markers_on = np.append(markers_on, np.where(cond)[0][-1])

    print("A:", As[j])
    print("sqrt(2)/2 - norm_inf:", -norm_inf[-1] + np.sqrt(2)/2)
    print("H2_norm:", H2_norm_plot[-1]**2)

    ax.plot(time_plot, H2_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markevery=markers_on, markerfacecolor='none') #


for j in range(2):
    #if As[j]!= 0.64375:
    norm_inf = norms_inf[j]
    cond_inf = norm_inf!=0
    norm_inf = norm_inf[cond_inf]

    H2_norm_plot = H2_norms[j]

    time_plot = times[j]
    #cond = H2_norm_plot!=0
    #H2_norm_plot = H2_norm_plot[cond]

    ax.plot(time_plot, H2_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markevery=1500, markerfacecolor='none') #

ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$|\Psi|_{H^2_h}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
#plt.ylim(10, 300)
ax.set_xlim(0, 3.5e-3)

#plt.yscale('log')
order = np.argsort(A_values)[::-1]
handles_sorted = [handles[i] for i in order] + [sol_line]
labels_sorted = ["A={}".format(round(A_values[i], 3)) for i in order]

ax.legend(handles_sorted, labels_sorted, fontsize=6, bbox_to_anchor=(1.05, 1))
ax.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

plt.savefig(path_save+"Blow_up_As_H2_norm_cut.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H2_norm.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
sol_lim = np.sqrt(2)/2+0.000085

fig = plt.figure()
#fig.patch.set_alpha(0)
ax = fig.add_subplot(111)

for j in range(2, len(As)):
    norm_inf = norms_inf[j]

    time_plot = times[j]
    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)  
    norm_inf = norm_inf[cond]

    H1_norm_plot = H1_norms[j][cond]

    #cond_1_markers = np.where(norm_inf_plot<sol_lim)[0]
    #mark_indices_1 = cond_1_markers[::3000]
    #cond_2_markers = np.where(norm_inf_plot>sol_lim)[0]
    #mark_indices_2 = cond_2_markers

    #mark_indices = np.append(mark_indices_1, mark_indices_2)

    #ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    #T = times[j][cond]#times[j][ind_exp]

    #mask = times[j] < T 
    time_plot = times[j][cond]#[mask]

    markers_on = np.arange(0, len(time_plot), 1500)
    markers_on = np.append(markers_on, np.where(cond)[0][-1])

    print("A:", As[j])
    print("sqrt(2)/2 - norm_inf:", -norm_inf[-1] + np.sqrt(2)/2)
    print("H2_norm:", H1_norm_plot[-1]**2)

    ax.plot(time_plot, H1_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markevery=markers_on, markerfacecolor='none') #


for j in range(2):
    #if As[j]!= 0.64375:
    norm_inf = norms_inf[j]
    cond_inf = norm_inf!=0
    norm_inf = norm_inf[cond_inf]

    H1_norm_plot = H1_norms[j]

    time_plot = times[j]
    #cond = H2_norm_plot!=0
    #H2_norm_plot = H2_norm_plot[cond]

    ax.plot(time_plot, H1_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markevery=1500, markerfacecolor='none') #

ax.set_xlabel(r"$t$")
ax.set_ylabel(r"$|\Psi|_{H^1_h}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
#plt.ylim(10, 300)
ax.set_xlim(0, 3.5e-3)

#plt.yscale('log')
#ax.legend(fontsize=6, bbox_to_anchor=(1.05, 1))
order = np.argsort(A_values)[::-1]
handles_sorted = [handles[i] for i in order] + [sol_line]
labels_sorted = ["A={}".format(round(A_values[i], 3)) for i in order]

ax.legend(handles_sorted, labels_sorted, fontsize=6, bbox_to_anchor=(1.05, 1))
ax.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

#ax.yaxis.tick_right()
#ax.yaxis.set_label_position("right")

plt.savefig(path_save+"Blow_up_As_H1_norm_cut.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H2_norm.pgf", dpi=300, bbox_inches = "tight")
plt.show()

Llega a $10^9$ pasado T por el esquema. La solución deja de ser razonable

In [ ]:

for j in range(2, len(As)):
    norm_inf = norms_inf[j]

    time_plot = times[j]
    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)  
    norm_inf = norm_inf[cond]

    H2_norm_plot = H2_norms[j][cond]

    #cond_1_markers = np.where(norm_inf_plot<sol_lim)[0]
    #mark_indices_1 = cond_1_markers[::3000]
    #cond_2_markers = np.where(norm_inf_plot>sol_lim)[0]
    #mark_indices_2 = cond_2_markers

    #mark_indices = np.append(mark_indices_1, mark_indices_2)

    #ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    #T = times[j][cond]#times[j][ind_exp]

    #mask = times[j] < T 
    time_plot = times[j][cond]#[mask]
    T = time_plot[-1]

    markers_on = np.arange(0, len(time_plot), 1500)
    markers_on = np.append(markers_on, np.where(cond)[0][-1])

    print("A:", As[j])
    print("sqrt(2)/2 - norm_inf:", -norm_inf[-1] + np.sqrt(2)/2)
    plt.plot(T-time_plot, H2_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markerfacecolor='none')


plt.xlabel(r"$T - t$")
plt.ylabel(r"$|\Psi|_{H^2}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
#plt.ylim(10, 300)
#plt.xlim(0, 3.5e-3)
plt.yscale("log")
plt.xscale("log")

plt.legend(fontsize=6)
#plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

#plt.savefig(path_save+"Blow_up_As_H2_norm.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H2_norm.pgf", dpi=300, bbox_inches = "tight")
#plt.ylim(1e1, 5e2)
plt.show()

In [ ]:
from scipy.stats import linregress
from scipy.optimize import curve_fit



for j in range(2, 9):
    norm_inf = norms_inf[j]

    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)

    #ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    time_plot = times[j][cond]
    dt = time_plot[1] - time_plot[0]
    T = time_plot[-1] #[ind_exp] #- 1e-5
    print(sci_notation(T))
    
    #sol_lim_up = np.sqrt(2)/2#+0.000085
    #sol_lim_down = 0#np.sqrt(2)/2-0.001
    
    H2_norm_plot = H2_norms[j][cond]
    H2_norm_plot = H2_norm_plot[H2_norm_plot!=0]
    #cond = H2_norm_plot!=0
    #H2_norm_plot = H2_norm_plot[cond]
    
    #time_plot = times[j]
    #cond_time = time_plot>0
    #time_plot = time_plot[cond]
    
    mask = np.logical_and(1e-3<time_plot, time_plot < T- 10*dt ) #times[j] < T- 10*dt
    x = np.log(-np.log(T - time_plot[mask]))
    y = np.log(H2_norm_plot[mask])#[mask]) # H2_norms[j][mask] #

    cond = True #np.logical_and(x<2.6, 2<x)
    
    xx = x#[cond]
    yy = y#[cond]

    slope, intercept = np.polyfit(x, y, 1)

    curve_pars, other = curve_fit(lambda t,C,p: C + p * t,  xx,  yy)
    C, p = curve_pars

    p_est = p
    C_est = np.exp(C)

    print("estimated p =", p_est)
    print("estimated c =", C_est)
    
    y_fit = C + p * xx
    
    plt.plot(x, y, 'o-', ms=1)#, label=r"$T^*=$"+str(np.log(-np.log(T))))
    plt.plot(xx, y_fit, '-', lw=3, label="A = "+str(round(As[j], 3)) + " p="+str(round(p, 3)))#, #label=f'linear regression, p={p_est:.4f}')

plt.legend(fontsize=6, bbox_to_anchor=(2, 1))
plt.xlabel(r'log(-log($T^*$ - t))')
plt.ylabel('log(H2_norm)')
#plt.legend()
plt.show()

In [ ]:
for j in range(3, 7):
    norm_inf = norms_inf[j]

    cond = norm_inf!=0
    cond = cond & (norm_inf<np.sqrt(2)/2)

    #ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    time_plot = times[j][cond]
    dt = times[j][1] - times[j][0]
    T = time_plot[-1] #[ind_exp] #- 1e-5
    print(sci_notation(T))
    
    #sol_lim_up = np.sqrt(2)/2#+0.000085
    #sol_lim_down = 0#np.sqrt(2)/2-0.001
    
    H2_norm_plot = H2_norms[j][cond]
    H2_norm_plot = H2_norm_plot[H2_norm_plot!=0]**2
    #cond = H2_norm_plot!=0
    #H2_norm_plot = H2_norm_plot[cond]
    
    #time_plot = times[j]
    #cond_time = time_plot>0
    #time_plot = time_plot[cond]
    
    mask = np.logical_and(T- 2000*dt<time_plot, time_plot < T- 10*dt ) #times[j] < T- 10*dt
    x = np.log(-np.log(T - time_plot[mask]))
    y = np.log(H2_norm_plot[mask])#[mask]) # H2_norms[j][mask] #

    cond = True #np.logical_and(x<2.6, 2<x)
    
    xx = x#[cond]
    yy = y#[cond]

    slope, intercept = np.polyfit(x, y, 1)

    curve_pars, other = curve_fit(lambda t,C,p: C + p * t,  xx,  yy)
    C, p = curve_pars

    p_est = p
    C_est = np.exp(C)

    print("estimated p =", p_est)
    print("estimated c =", C_est)
    
    y_fit = C + p * xx
    
    plt.plot(x, y, 'o-', ms=1)#, label=r"$T^*=$"+str(np.log(-np.log(T))))
    plt.plot(xx, y_fit, '-', lw=3, label="A = "+str(round(As[j], 3)) + " p="+str(round(p, 3)))#, #label=f'linear regression, p={p_est:.4f}')

plt.legend(fontsize=6, bbox_to_anchor=(2, 1))
plt.xlabel(r'log(-log($T^*$ - t))')
plt.ylabel('log(H2_norm)')
#plt.legend()
plt.show()

## Double points in time

In [ ]:
As = [0.64375, 0.65, 0.65625, 0.6609375]

N = 160_000
Nt = 2*64_221
tf = 2.5e-3
sigma = 1/10

norms_inf = []
H1_norms = []
H2_norms = np.full(len(As), fill_value=None)
energies = []
masses = []
times = []

for j in range(len(As)):
    folder_name = path_double + 'blow_up_N_{N}_Nt_{Nt}_sigma_{s}_A_{A}/'.format(N=N, Nt=Nt, s=sigma, A=round(As[j], 3))

    H1_file = gzip.GzipFile(folder_name + "H1_norm.npy.gz", "r")
    H1_norms += [np.load(H1_file)]

    #if As[j]!= 0.64375:
    H2_file = gzip.GzipFile(folder_name + "H2_norm.npy.gz", "r")
    H2_norms[j] = np.load(H2_file)

    norm_file = gzip.GzipFile(folder_name + "norm_inf.npy.gz", "r")
    norms_inf += [np.load(norm_file)]

    energy_file = gzip.GzipFile(folder_name + "energy.npy.gz", "r")
    energies += [np.load(energy_file)]

    mass_file = gzip.GzipFile(folder_name + "mass.npy.gz", "r")
    masses += [np.load(mass_file)]

    time_file = gzip.GzipFile(folder_name + "time.npy.gz", "r")
    times += [np.load(time_file)]

In [ ]:
for j in range(len(As)):
    # t explosion
    norm_inf = norms_inf[j]
    ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    T = times[j][ind_exp] #- 1e-5
    print(T)
    
    #sol_lim_up = np.sqrt(2)/2#+0.000085
    #sol_lim_down = 0#np.sqrt(2)/2-0.001
    
    
    H2_norm_plot = H2_norms[j]
    cond = H2_norm_plot!=0
    H2_norm_plot = H2_norm_plot[cond]
    
    time_plot = times[j]
    #cond_time = time_plot>0
    time_plot = time_plot[cond]
    
    norm_inf = norms_inf[j]
    norm_inf = norm_inf[norm_inf!=0]
    #cond_inf = np.logical_and(sol_lim_down<norm_inf, norm_inf<sol_lim_up)
    
    plt.plot(T-time_plot, H2_norm_plot,label="A={}".format(round(As[j], 3)), ms=3.5, markerfacecolor='none') #

plt.xlabel(r"$T - t$")
plt.ylabel(r"$|\Psi|_{H^2}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
#plt.ylim(10, 300)
#plt.xlim(0, 3.5e-3)
plt.yscale("log")
plt.xscale("log")

plt.legend(fontsize=6)
#plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

#plt.savefig(path_save+"Blow_up_As_H2_norm.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H2_norm.pgf", dpi=300, bbox_inches = "tight")
plt.ylim(1e1, 5e2)
plt.show()

In [ ]:
from scipy.stats import linregress
from scipy.optimize import curve_fit

j = 4
for j in range(len(As)):
    norm_inf = norms_inf[j]
    ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    T = times[j][ind_exp] #- 1e-5
    print(T)
    
    #sol_lim_up = np.sqrt(2)/2#+0.000085
    #sol_lim_down = 0#np.sqrt(2)/2-0.001
    
    
    H2_norm_plot = H2_norms[j]
    cond = H2_norm_plot!=0
    H2_norm_plot = H2_norm_plot[cond]
    
    time_plot = times[j]
    #cond_time = time_plot>0
    time_plot = time_plot[cond]
    
    mask = times[j] < T
    x = np.log(-np.log(T - times[j][mask]))
    cond = np.logical_and(x<2.6, 2<x)
    xx = x[cond]
    y = np.log(H2_norms[j][mask])
    yy = y[cond]

    #slope, intercept = np.polyfit(x, y, 1)

    curve_pars, other = curve_fit(lambda t,C,p: C + p * t,  xx,  yy)
    C, p = curve_pars

    p_est = p
    C_est = np.exp(C)

    print("estimated p =", p_est)
    print("estimated c =", C_est)
    
    y_fit = C + p * xx
    
    plt.plot(x, y, 'o-', ms=1)#, label=r"$T^*=$"+str(np.log(-np.log(T))))
    plt.plot(xx, y_fit, '-', lw=3, label="A = "+str(round(As[j], 3)) + " p="+str(round(p, 3)))#, #label=f'linear regression, p={p_est:.4f}')

plt.legend()
plt.xlabel(r'log(-log($T^*$ - t))')
plt.ylabel('log(H2_norm)')
#plt.legend()
plt.show()

In [ ]:
from scipy.stats import linregress
from scipy.optimize import curve_fit


for j in range(len(As)):
    norm_inf = norms_inf[j]
    ind_exp = np.where(np.isclose(norm_inf, np.sqrt(2)/2, rtol=1e-5, atol=1e-10))[0][0]
    dt = times[j][1] - times[j][0]
    T = times[j][ind_exp] #- 1e-5
    print(sci_notation(T))
    
    #sol_lim_up = np.sqrt(2)/2#+0.000085
    #sol_lim_down = 0#np.sqrt(2)/2-0.001
    
    
    H2_norm_plot = H2_norms[j]
    cond = H2_norm_plot!=0
    H2_norm_plot = H2_norm_plot[cond]
    
    time_plot = times[j]
    #cond_time = time_plot>0
    time_plot = time_plot[cond]
    
    mask = np.logical_and(T- 4000*dt<times[j], times[j] < T- 20*dt ) #times[j] < T- 10*dt
    x = np.log(-np.log(T - times[j][mask]))
    y = np.log(H2_norms[j][mask]) # H2_norms[j][mask] #

    cond = True #np.logical_and(x<2.6, 2<x)
    
    xx = x#[cond]
    yy = y#[cond]

    slope, intercept = np.polyfit(x, y, 1)

    curve_pars, other = curve_fit(lambda t,C,p: C + p * t,  xx,  yy)
    C, p = curve_pars

    p_est = p
    C_est = np.exp(C)

    print("estimated p =", p_est)
    print("estimated c =", C_est)
    
    y_fit = C + p * xx
    
    plt.plot(x, y, 'o-', ms=1)#, label=r"$T^*=$"+str(np.log(-np.log(T))))
    plt.plot(xx, y_fit, '-', lw=3, label="A = "+str(round(As[j], 3)) + " p="+str(round(p, 3)))#, #label=f'linear regression, p={p_est:.4f}')

plt.legend(fontsize=6, bbox_to_anchor=(2, 1))
plt.xlabel(r'ln(-ln($T^*$ - t))')
plt.ylabel(r"$\ln(|\Psi|_{H^2})$", rotation='horizontal', ha='right')
#plt.legend()
plt.show()

En esa celda, como defines

$$
x=\log\!\big(-\log(T-t)\big),
$$

cuando $t \to T^{-}$ se cumple:

1. $T-t \to 0^{+}$
2. $-\log(T-t)\to \infty$
3. $\log(-\log(T-t))\to \infty$

Entonces, en ese gráfico debes mirar la parte derecha (valores grandes de $x$). Ahí estás más cerca de $T$.

Referencia rápida según variable en eje $x$:

1. Si graficas $T-t$ (o $\log(T-t)$): cerca de $T$ está a la izquierda.
2. Si graficas $\log(-\log(T-t))$: cerca de $T$ está a la derecha.

## Agregar: Cambio en el cuarto decimal usando el doble de puntos en tiempo

In [ ]:
def D2_neumann_centered(N, h):
    """
    Matriz de segunda derivada con condiciones de Neumann homogéneas,
    usando diferencias centradas también en los bordes.

    La matriz resultante es:

        1/h^2 * [ -2   2   0  ...  0
                   1  -2   1  ...  .
                   0   1  -2  ...  0
                   .   .   .  ...  1
                   0  ...  0   2  -2 ]

    Parámetros
    ----------
    N : int
        Número de puntos de malla.
    h : float
        Paso espacial.

    Retorna
    -------
    scipy.sparse.csr_matrix
        Matriz dispersa NxN.
    """
    e = np.ones(N)

    D2 = diags(
        diagonals=[e, -2.0 * e, e],
        offsets=[-1, 0, 1],
        shape=(N, N),
        format="lil"
    )

    # Ajustes de borde para Neumann homogénea con esquema centrado
    #D2[0, 1] = 2.0
    #D2[-1, -2] = 2.0

    return D2.tocsr() / h**2

In [ ]:
import numpy as np
from Eigenvalues import *

def H2_norm_num(u, h):
    N = len(u)
    Dxx = D2_neumann_centered(N, h)
    Dxxu = Dxx @ u
    return np.sqrt(np.sum(h * np.abs(Dxxu)**2)) 
#np.sqrt(h) * np_norm(np.gradient(np.gradient(u, h), h), 2)

In [ ]:
def seminorm_H2_f_eps(L, eps, con_raiz=True):

    semi2 = (
        L * (3 * L**2 + 5 * eps**2) / (4 * (L**2 + eps**2)**2)
        + (3 / (4 * eps)) * np.arctan(L / eps)
    )

    return np.sqrt(semi2) if con_raiz else semi2

from scipy.integrate import simpson

def seminorm_H2_f_eps_2(L, eps):

    def integrand(x):
        r = np.sqrt(x*x + eps)
        sech2 = 1.0 / np.cosh(r)**2
        term = eps / (x*x + eps)**1.5 - 2.0 * x*x / (x*x + eps) * np.tanh(r)
        return (sech2**2) * (term**2)

    x = np.linspace(0, L, 1000)
    y = integrand(x)
    value = simpson(y, x, dx=x[1] - x[0])
    return np.sqrt(2.0 * value)

In [ ]:
L = 10
x = np.linspace(-L, L, 100_000)
eps = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10]
H2 = []
seminorm = []

for e in eps:
    u = f_eps_2(x, e)
    h = x[1] - x[0]
    norm_H2 = H2_norm(u, h)
    H2.append(norm_H2)
    seminorm.append(seminorm_H2_f_eps_2(L, e))


In [ ]:
print("numeric H2:", H2)
print("real seminorm H2:", seminorm)

In [ ]:
import numpy as np
from Eigenvalues import *

def H1_seminorm_num(u, h):
    N = len(u)
    Dx = D1_neumann_centered(N, h)
    Dxu = Dx @ u
    return np.sqrt(np.sum(h * np.abs(Dxu)**2)) 

def H2_seminorm_num(u, h):
    N = len(u)
    Dxx = D2_neumann_centered(N, h)
    Dxxu = Dxx @ u
    return np.sqrt(np.sum(h * np.abs(Dxxu)**2)) 
#np.sqrt(h) * np_norm(np.gradient(np.gradient(u, h), h), 2)

In [ ]:
def f_epsilon(x, epsilon):
    x = np.asarray(x, dtype=float)

    y = np.zeros_like(x)
    mask = np.abs(x) <= epsilon
    y[mask] = (1.0 - (x[mask] ** 2) / (epsilon ** 2)) ** 3
    return y


def l2_norm(epsilon):
    return np.sqrt((2048.0 / 3003.0) * epsilon)



def h1_seminorm(epsilon):
    return np.sqrt((1024.0 / 385.0) * epsilon ** (-1))


def h2_seminorm(epsilon):
    return np.sqrt((1024.0 / 35.0) * epsilon ** (-3))


In [ ]:
L = np.pi
x = np.linspace(-L, L, int(2e5))
dx = x[1] - x[0]
plt.plot(x, f_epsilon(x, 1e-4), ms=0)
plt.xlim(-1, 1)
plt.show()
print("dx:", dx)

In [ ]:
L = np.pi
N = int(1000)
x = np.linspace(-L, L, N)
dx = x[1] - x[0]

Dxx = D2_neumann_centered(N, dx)
Dxxu = Dxx @ f_epsilon(x, 1e-3)

plt.plot(x, Dxxu, ms=0)
plt.xlim(-1, 1)
plt.show()
print("dx:", dx)


In [ ]:
2*np.pi/160_000

Añadir al apéndice: Dificultad de que intente regularizar cosas que no lo son

In [ ]:
L = 1
x = np.linspace(-L, L, 1_000_000)
eps = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9, 1e-10]
seminorm_H1_num_ = []
seminorm_H2_num_ = []

seminorm_H1_ = []
seminorm_H2_ = []

for e in eps:
    u = f_epsilon(x, e)
    h = x[1] - x[0]
    norm_H1 = H1_seminorm_num(u, h)
    norm_H2 = H2_seminorm_num(u, h)

    seminorm_H1_num_.append(norm_H1)
    seminorm_H2_num_.append(norm_H2)

    seminorm_H1_.append(h1_seminorm(e))
    seminorm_H2_.append(h2_seminorm(e))


In [ ]:
print("numeric H1 seminorm:", seminorm_H1_num_)
print("theoretical H1 seminorm:", seminorm_H1_)

In [ ]:
print("numeric H2 seminorm:", seminorm_H2_num_)
print("theoretical H2 seminorm:", seminorm_H2_)

# Other estimations

In [ ]:
from scipy.linalg import lstsq

j = 4
time = times[j]
H2_norm = H2_norms[j]

# Asumimos:
# time, H2_norm, T ya definidos

mask = time < T

tau = T - time[mask]
y = np.log(H2_norm[mask])

x1 = np.log(tau)
x2 = np.log(np.log(1.0 / tau))

# matriz de diseño: y ~ a + b*x1 + c*x2
A = np.column_stack([np.ones_like(x1), x1, x2])

coef, _, _, _ = lstsq(A, y)

a, b, c = coef
p_est = -b
q_est = -c
C_est = np.exp(a)

print("p estimado =", p_est)
print("q estimado =", q_est)
print("C estimada =", C_est)

# ajuste
y_fit = A @ coef

# R^2
ss_res = np.sum((y - y_fit)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r2 = 1 - ss_res / ss_tot

print("R^2 =", r2)

In [ ]:
j = 4
time = times[j]
H2_norm = H2_norms[j]

mask = time < T
t = time[mask]
tau = T - t

H2_fit = C_est * tau**(-p_est) * (np.log(1.0 / tau))**(-q_est)

plt.semilogy(t, H2_norm[mask], 'o', ms=3, label='datos')
plt.semilogy(t, H2_fit, '-', lw=2, label='ajuste')
plt.xlabel('t')
plt.ylabel('H2_norm')
plt.legend()
plt.show()

In [ ]:
j = 4
time = times[j]
H2_norm = H2_norms[j]

mask = time < T
tau = T - time[mask]
y_data = H2_norm[mask]

# ordenar por tau creciente o decreciente, para que la curva salga prolija
idx = np.argsort(tau)
tau = tau[idx]
y_data = y_data[idx]

# modelo ajustado
y_fit = C_est * tau**(-p_est) * (np.log(1.0 / tau))**(-q_est)

plt.loglog(tau, y_data, 'o', ms=2, label='datos')
plt.loglog(tau, y_fit, '-', label='aproximación')
plt.xlabel('T - t')
plt.ylabel('H2_norm')
plt.legend()
plt.show()


In [ ]:
for j in range(len(As)):
    norm_inf = norms_inf[j]
    cond_inf = norm_inf!=0
    norm_inf = norm_inf[cond_inf]

    cond_1_markers = np.where(norm_inf<sol_lim)[0]
    mark_indices_1 = cond_1_markers[::3500]
    cond_2_markers = np.where(norm_inf>sol_lim)[0]
    mark_indices_2 = cond_2_markers
    mark_indices = np.append(mark_indices_1, mark_indices_2)
    
    H1_norm_plot = H1_norms[j]
    time_plot = times[j]
    cond = H1_norm_plot!=0
    plt.plot(time_plot[cond], H1_norm_plot[cond], label="A={}".format(round(As[j], 3)), markevery=mark_indices, markerfacecolor='none')

plt.xlabel(r"$t$")
plt.ylabel(r"$|\Psi|_{H^1}$", rotation='horizontal', ha='right')
#plt.hlines(np.sqrt(2)/2, 0, 3.5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.ylim(1.83, 3)
plt.xlim(0, 3.5e-3)
plt.legend(fontsize=6)
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

plt.savefig(path_save+"Blow_up_As_H1_norm.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Blow_up_As_H1_norm.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## $A = 0.65$

In [ ]:
L = np.pi
sols = []

A_065_folder = path + "blow_up_N_200000_Nt_64300_sigma_0.1_A_0.65/"
t_file = gzip.GzipFile(A_065_folder + "time.npy.gz", "r")
t = np.load(t_file)

mass_file = gzip.GzipFile(A_065_folder+"mass.npy.gz", "r")
mass = np.load(mass_file)

H1_norm_file = gzip.GzipFile(A_065_folder+"H1_norm.npy.gz", "r")
H1_norm = np.load(H1_norm_file)

H2_norm_file = gzip.GzipFile(A_065_folder+"H2_norm.npy.gz", "r")
H2_norm = np.load(H2_norm_file)

t_file = gzip.GzipFile(A_065_folder+"time.npy.gz", "r")
t = np.load(t_file)

energy_file = gzip.GzipFile(A_065_folder+"energy.npy.gz", "r")
energy = np.load(energy_file)

t_plot = []

N = 200_000
Nt = 64_300

first_N_times = my_lin(0, 64_200, 100, 0.5)[1:-1]
second_N_times = np.arange(64_200, 64_256)
N_times = np.append(first_N_times, second_N_times)

index = 0

for j in range(N_times[-1]-1):
    if (j+2)==N_times[index]:
        sol_file = gzip.GzipFile(A_065_folder + "w_"+f"{j+2:010d}"+".npy.gz", "r")
        sol = np.load(sol_file)
        sols += [sol]
        t_plot += [t[j]]
        index += 1

#sol_file = gzip.GzipFile(folder + "w_0000064256.npy.gz", "r")
#sol = np.load(sol_file)
#sols += [sol]

t_plot = np.array(t_plot)
sols = np.array(sols)
sols = np.stack(sols)

In [ ]:
x = np.linspace(-L, L, N)

fig = plt.figure()
fig.patch.set_alpha(0)
ax = fig.add_subplot(111)

for j in range(0, len(sols)-1, 8):
    ax.plot(x, np.abs(sols[j]), ls="-", marker="none", lw=.5)

ax.hlines(np.sqrt(2)/2, -0.035, 0.035, ls="--", color="black", alpha=0.6, label=r"$\sqrt{2}/2$")

ax.annotate(
    '',
    xy=(0.0195, np.sqrt(2)/2-0.01),
    xytext=(0.0195, 0.66),
    arrowprops=dict(arrowstyle='->', lw=1),
    xycoords='data'
)

ax.text(
    0.021, 0.675,
    r'$t$',
    fontsize=12,
    ha='left',
    va='center'
)

ax.legend()
#plt.xlim(-0.3, 0.3)
ax.set_xlim(-0.035, 0.035)
ax.set_ylim(0.6, 0.75)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$|\Psi|$", rotation='horizontal', ha='left')

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

plt.savefig(path_save+"blow_up_evol_2D_poster.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
s = 0.1
A = 0.65
L = np.pi
x = np.linspace(-L, L, N)

labels = [r"$x$", r"$t$", r"$|\Psi|^2$"]
name_pdf = path_save+"blow_up_sigma_{s}_A_{A}.pdf".format(s=s, A=A)
#name_pgf = path_save+"blow_up_sigma_{s}_A_{A}.pgf".format(s=s, A=A)

#line_plot_3D(sols, x, t_plot, name_pdf, labels, L=0.2, Nt_plot=100, x_rot=-90, z_pad=-27, y_pad=-8, sci=True)
#line_plot_3D(sols, x, t_plot, name_pgf, labels, L=0.2, Nt_plot=100, x_rot=-90, z_pad=-27, y_pad=-8, sci=True)

In [ ]:
idx_blow_up = np.where(np.abs(energy - energy[0])>1)[0][0] #np.argmax(np.gradient(energies[j]))
ts_blow = 0.002158186420517196 #t[idx_blow_up]

In [ ]:
cond = H2_norm!=0
y_upper_lim = 200

plt.plot(t[cond], H2_norm[cond], ls="-", markevery=500, markerfacecolor="none")
plt.vlines(ts_blow, 0, y_upper_lim, ls="--", color="black", alpha=0.5,
           label=r"$T^* \approx $ {}".format(sci_notation(ts_blow, n_decimals=4)))
plt.legend()
#plt.yscale("log")
plt.ylim(20, y_upper_lim)
plt.xlim(1.5e-3, 2.25e-3)
plt.xlabel(r"$t$")
plt.ylabel(r"$|\Psi|_{H^2}$", rotation='horizontal', ha='right')
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

plt.savefig("Img/Heaven/blow_up_time.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig("Img/Heaven/blow_up_time.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
cond = energy!=0
y_upper_lim = 200

plt.plot(t[cond], energy[cond]-energy[0], markevery=2500, markerfacecolor="none")
#plt.vlines(ts_blow, 0, y_upper_lim, ls="--", color="black", alpha=0.5,
#           label=r"$t_b \approx $ {}".format(sci_notation(ts_blow, n_decimals=4)))
#plt.legend()
#plt.yscale("log")
plt.ylim(-1, 0.5)
#plt.xlim(1.5e-3, 2.25e-3)
plt.xlabel(r"$t$")
plt.ylabel(r"$E(t)-E(0)$")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")

#plt.savefig("Img/Heaven/blow_up_time.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig("Img/Heaven/blow_up_time.pgf", dpi=300, bbox_inches = "tight")
plt.show()

# $A^*$ estimation 

In [ ]:
path = "../../../scripts/Results/Blow_up/A_est/"
path_sigma_005 = path + "blow_up_A_est_sigma_0.05/"
path_sigma_01 = path + "blow_up_A_est_sigma_0.1/"
path_sigma_02 = path + "blow_up_A_est_sigma_0.2/"
path_sigma_05 = path + "blow_up_A_est_sigma_0.5/"

In [ ]:
A_guesses_005_file = gzip.GzipFile(path_sigma_005 + "A_guesses.npy.gz", "r")
A_guesses_005 = np.load(A_guesses_005_file)

A_guesses_01_file = gzip.GzipFile(path_sigma_01 + "A_guesses.npy.gz", "r")
A_guesses_01 = np.load(A_guesses_01_file)

A_guesses_02_file = gzip.GzipFile(path_sigma_02 + "A_guesses.npy.gz", "r")
A_guesses_02 = np.load(A_guesses_02_file)

A_guesses_05_file = gzip.GzipFile(path_sigma_05 + "A_guesses.npy.gz", "r")
A_guesses_05 = np.load(A_guesses_05_file)

In [ ]:
pt = 1./72.27 # Hundreds of years of history... 72.27 points to an inch.

jour_sizes = {"PRD": {"onecol": 246.*pt, "twocol": 510.*pt},
              "CQG": {"onecol": 374.*pt}, # CQG is only one column
              "DEF": {"onecol": 455.24408*pt}
             }

my_width = jour_sizes["DEF"]["onecol"]
# Our figure's aspect ratio
golden = (1 + 5 ** 0.5) / 2

default_size = plt.rcParams["figure.figsize"]
plt.figure(figsize = (my_width, default_size[1])) #my_width/golden))

plt.plot(A_guesses_005[A_guesses_005!=0], label=r"$\sigma = 0.05$", markerfacecolor="none")
#plt.annotate('%0.5f' % A_guesses_005[A_guesses_005!=0][-1], xy=(1, A_guesses_005[A_guesses_005!=0][-1]), xytext=(3, -3), 
#             xycoords=('axes fraction', 'data'), textcoords='offset points')

plt.plot(A_guesses_01[A_guesses_01!=0], label=r"$\sigma = 0.1$", markerfacecolor="none")
#plt.annotate('%0.5f' % A_guesses_01[A_guesses_01!=0][-1], xy=(1, A_guesses_01[A_guesses_01!=0][-1]), xytext=(3, -3), 
#             xycoords=('axes fraction', 'data'), textcoords='offset points')

plt.plot(A_guesses_02[A_guesses_02!=0], label=r"$\sigma = 0.2$", markerfacecolor="none")
#plt.annotate('%0.5f' % A_guesses_02[A_guesses_02!=0][-1], xy=(1, A_guesses_02[A_guesses_02!=0][-1]), xytext=(3, 1), 
#             xycoords=('axes fraction', 'data'), textcoords='offset points')

plt.plot(A_guesses_05[A_guesses_05!=0], label=r"$\sigma = 0.5$", markerfacecolor="none")
#plt.annotate('%0.5f' % A_guesses_05[A_guesses_05!=0][-1], xy=(1, A_guesses_05[A_guesses_05!=0][-1]), xytext=(3, 0), 
#             xycoords=('axes fraction', 'data'), textcoords='offset points')

plt.legend()
plt.ylabel(r"$A^*$", rotation='horizontal', ha='right')
plt.xlabel("Iterations")
plt.xticks(range(0, 21, 2))

plt.savefig(path_save_local+"Gauss_As/A_est.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save_local+"Gauss_As/A_est.pgf", bbox_inches = "tight")
plt.show()

print("Last iteration value sigma = 0.05: ", A_guesses_005[A_guesses_005!=0][-1])
print("Last iteration value sigma = 0.1:  ", A_guesses_01[A_guesses_01!=0][-1])
print("Last iteration value sigma = 0.2:  ", A_guesses_02[A_guesses_02!=0][-1])
print("Last iteration value sigma = 0.5:  ", A_guesses_05[A_guesses_05!=0][-1])

In [ ]:
A_guesses_005[A_guesses_005!=0]

## $A^*$ test

### $\sigma = 0.1$ without blow up

In [ ]:
# No Blow up
eps = 1e-3
A =  A_guesses_01[A_guesses_01!=0][-1] -eps #1 #0.625 #0.65
sigma = 1/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 5_000 #10_000
tf = 5e-3
pars = {"k":k}

w_sigma_1_A_nbl = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_sigma_1_A_nbl.set_init_condition(gaussian)
w_sigma_1_A_nbl.solve()#method="fixed point")

### $\sigma = 0.1$ with blow up

In [ ]:
# No Blow up
eps = 1e-3
A =  A_guesses_01[A_guesses_01!=0][-1] + eps #1 #0.625 #0.65
sigma = 1/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 2_000 #10_000
tf = 3.5e-3
pars = {"k":k}

w_sigma_1_A_bl = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_sigma_1_A_bl.set_init_condition(gaussian)
w_sigma_1_A_bl.solve()#method="fixed point")

In [ ]:
w_sigma_1_A_nbl_max = np_norm(w_sigma_1_A_nbl.w, np.inf, axis=1)
w_sigma_1_A_bl_max = np_norm(w_sigma_1_A_bl.w, np.inf, axis=1)

In [ ]:
cond_nbl = w_sigma_1_A_nbl_max > 0
cond_bl = w_sigma_1_A_bl_max > 0

plt.plot(w_sigma_1_A_nbl.t[cond_nbl], w_sigma_1_A_nbl_max[cond_nbl], markevery=100, ms=2, label=r"$A = A^* - \varepsilon$")
plt.plot(w_sigma_1_A_bl.t[cond_bl], w_sigma_1_A_bl_max[cond_bl], markevery=50, ms=2, label=r"$A = A^* + \varepsilon$")
plt.hlines(np.sqrt(2)/2, 0, 5e-3, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.legend()
plt.ylim(0.6, 1)
plt.ylabel(r"$||\Psi||_\infty$", rotation='horizontal', ha='right')
plt.xlabel(r"$t$")
plt.ticklabel_format(style='sci',scilimits=(-3,4),axis="x")
plt.xlim(0, 5e-3)
#plt.xlim(2, 3.8e-3)
plt.savefig(path_save_local+"Gauss_As/Blow_up_A_star_sigma_0_1.pdf", dpi=300, bbox_inches = "tight")
plt.show()

### $\sigma = 0.2$ without blow up

In [ ]:
# No Blow up
eps = 1e-3
A =  A_guesses_02[A_guesses_02!=0][-1] -eps #1 #0.625 #0.65
sigma = 2/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 5_000 #10_000
tf = 2.5e-2
pars = {"k":k}

w_sigma_2_A_nbl = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_sigma_2_A_nbl.set_init_condition(gaussian)
w_sigma_2_A_nbl.solve()#method="fixed point")

### $\sigma = 0.2$ with blow up

In [ ]:
# No Blow up
eps = 1e-3
A =  A_guesses_02[A_guesses_02!=0][-1] + eps #1 #0.625 #0.65
sigma = 2/10

def gaussian(x, sigma=sigma):
    output = A * np.exp(-x**2/(2 * sigma**2))
    return output

In [ ]:
L = np.pi
N = 10_000
k = 1
Nt = 2_000 #10_000
tf = 2e-2
pars = {"k":k}

w_sigma_2_A_bl = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_sigma_2_A_bl.set_init_condition(gaussian)
w_sigma_2_A_bl.solve()#method="fixed point")

In [ ]:
w_sigma_2_A_nbl_max = np_norm(w_sigma_2_A_nbl.w, np.inf, axis=1)
w_sigma_2_A_bl_max = np_norm(w_sigma_2_A_bl.w, np.inf, axis=1)

In [ ]:
cond_2_nbl = w_sigma_2_A_nbl_max > 0
cond_2_bl = w_sigma_2_A_bl_max > 0

plt.plot(w_sigma_2_A_nbl.t[cond_2_nbl], w_sigma_2_A_nbl_max[cond_2_nbl], markevery=50, ms=2, label=r"$A = A^* - \varepsilon$")
plt.plot(w_sigma_2_A_bl.t[cond_2_bl], w_sigma_2_A_bl_max[cond_2_bl], markevery=50, ms=2, label=r"$A = A^* + \varepsilon$")

plt.hlines(np.sqrt(2)/2, 0, 2e-2, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.legend()
plt.ylim(0.6, 1)
plt.ylabel(r"$||\Psi||_\infty$", rotation='horizontal', ha='right')
plt.xlabel(r"$t$")
plt.ticklabel_format(style='sci',scilimits=(-2,4),axis="x")
plt.xlim(0, 2e-2)
#plt.xlim(2, 3.8e-3)
plt.savefig(path_save_local+"Gauss_As/Blow_up_A_star_sigma_0_2.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
cond_1_nbl = w_sigma_1_A_nbl_max > 0
cond_1_bl = w_sigma_1_A_bl_max > 0

cond_2_nbl = w_sigma_2_A_nbl_max > 0
cond_2_bl = w_sigma_2_A_bl_max > 0

plt.plot(w_sigma_1_A_nbl.t[cond_1_nbl], w_sigma_1_A_nbl_max[cond_1_nbl], markevery=50, ms=2, label=r"$\sigma=0.1$ and $A = A^* - \varepsilon$")
plt.plot(w_sigma_1_A_bl.t[cond_1_bl], w_sigma_1_A_bl_max[cond_1_bl], markevery=50, ms=2, label=r"$\sigma=0.1$ and $A = A^* + \varepsilon$")

plt.plot(w_sigma_2_A_nbl.t[cond_2_nbl], w_sigma_2_A_nbl_max[cond_2_nbl], markevery=50, ms=2, label=r"$\sigma=0.2$ and $A = A^* - \varepsilon$")
plt.plot(w_sigma_2_A_bl.t[cond_2_bl], w_sigma_2_A_bl_max[cond_2_bl], markevery=50, ms=2, label=r"$\sigma=0.2$ and $A = A^* + \varepsilon$")

plt.hlines(np.sqrt(2)/2, 0, 2e-2, ls="--", color="gray", label=r"$\sqrt{2}/2$")
plt.legend()
plt.ylim(0.6, 1)
plt.ylabel(r"$|\Psi|_\infty$")
plt.xlabel(r"$t$")
plt.ticklabel_format(style='sci',scilimits=(-2,4),axis="x")
plt.xlim(0, 2e-2)
#plt.xlim(2, 3.8e-3)
#plt.savefig(path_save_local+"Gauss_As/Blow_up_A_star_sigma_0_2.pdf", dpi=300, bbox_inches = "tight")
plt.show()